[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhijeet1520/langgraph-workshop/blob/main/notebooks/level-1-basics/04_chat_tool_graph.ipynb)

# Chat + Tool Graph

**Level 1** · Part of the [LangGraph Workshop](https://github.com/Abhijeet1520/langgraph-workshop).

> **Source:** migrated from `snehotosh/langgraph-notebooks` ([original](https://github.com/snehotosh/langgraph-notebooks/blob/main/3_build_chat_tool_graph.ipynb)) — see [PROVENANCE.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/PROVENANCE.md).

**Provider setup:** set `OPENAI_API_KEY` / `OPENAI_API_BASE` / `OPENAI_MODEL` first — [docs/setup/providers.md](https://github.com/Abhijeet1520/langgraph-workshop/blob/main/docs/setup/providers.md). The first cells install requirements and verify the connection.

## Build a `Langgraph` with `llm`, `tool calling` and `binding`

In [ ]:
# --- Workshop provider config (edit ONLY these 3 values) ---------------------
import os

os.environ.setdefault("OPENAI_API_KEY", "")   # <- paste your key
os.environ.setdefault("OPENAI_API_BASE", "")  # <- paste your base URL
os.environ.setdefault("OPENAI_MODEL", "openai/gpt-4.1-mini")  # <- your model id

# Verify
for _v in ("OPENAI_API_KEY", "OPENAI_API_BASE", "OPENAI_MODEL"):
    print(_v, "=", "set" if os.getenv(_v) else "**MISSING — see docs/setup/providers.md**")
# -----------------------------------------------------------------------------

In [ ]:
from langchain_openai import ChatOpenAI
import os

def make_llm(**overrides):
    """Provider-agnostic LLM factory used across the workshop."""
    cfg = dict(
        model=os.environ["OPENAI_MODEL"],
        api_key=os.environ["OPENAI_API_KEY"],
        base_url=os.environ["OPENAI_API_BASE"],
        temperature=0,
    )
    cfg.update(overrides)
    return ChatOpenAI(**cfg)

llm = make_llm()

# Connection check (1 token)
resp = llm.invoke("Say OK")
print("✅ LLM reachable:", resp.content[:60])


In [ ]:
from typing import TypedDict, Annotated, Any

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, AnyMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

from langgraph.graph import START, END, StateGraph, add_messages
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

from IPython.display import display, Image

from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("CLAW_OMNI_API_KEY")
os.environ["BASE"]

#### Model

In [ ]:
llm = make_llm()

#### State

In [ ]:
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

#### Tools

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Add a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def sub(a: int, b: int) -> int:
    """Subtract b from a.

    Args:
        a: first int
        b: second int
    """
    return a - b

def div(a: int, b: int) -> int:
    """Divide a by b.

    Args:
        a: first int
        b: second int
    """
    return a // b


In [ ]:
'''
It wraps your base LLM (ChatOpenAI in this case) and gives it structured awareness of the tools (functions) you provide — so the model can:
1) Know which tools exist (name, docstring, parameters)
2) Know how to call them using structured JSON arguments
3) Produce tool call requests instead of just text
Important: It Does Not Execute the Tool
'''
llm_with_tools = llm.bind_tools([multiply, add, sub, div])

In [ ]:
'''
When you call llm_with_tools.invoke(...), the LLM decides whether to call a tool or not — but it does not automatically execute the tool.
'''
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
# Use built-in ToolNode to handle execution of the tool calls
'''
ToolNode is a prebuilt LangGraph component that automatically:
1) Listens for tool call requests emitted by your LLM
2) Executes those tool functions with the provided arguments
3) Returns the results back to the graph as messages
'''
tool_node = ToolNode(tools=[multiply, add, sub, div])

#### Build graph

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node("tool_calling_llm", tool_calling_llm)
graph.add_node("tools", tool_node)

graph.add_edge(START, "tool_calling_llm")
graph.add_edge("tool_calling_llm", "tools")
graph.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
graph.add_edge("tools", END)
app = graph.compile()

In [ ]:
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
result1 = app.invoke({"messages": HumanMessage(content="multiply 8 and 2")})

In [ ]:
for m in result1['messages']:
    m.pretty_print()

In [ ]:
result2 = app.invoke({"messages": HumanMessage(content="what is the total population of India?")})

for m in result2['messages']:
    m.pretty_print()